# Thompson-Konstruktion für reguläre Ausdrücke

Willkommen zum Workshop! In diesem Notebook lernst du, wie man die **Thompson-Konstruktion** implementiert. Das ist ein eleganter Algorithmus, der einen regulären Ausdruck (den wir zuerst in einen sogenannten AST - Abstract Syntax Tree - zerlegen) in einen nicht-deterministischen endlichen Automaten (NFA) verwandelt.

Keine Sorge, wenn du noch nie von Automaten gehört hast – wir gehen alles Schritt für Schritt durch. Wir nutzen zwei kleine Python-Pakete:
1. `ast_parser`: Liest deinen Regex-String und baut daraus einen Strukturbaum (AST).
2. `automaton`: Liefert uns die Bausteine für den Automaten (Zustände und Übergänge) und einen Executor, um zu testen, ob der Automat funktioniert.

In [ ]:
!pip install graphviz
import sys

sys.path.append('.')

In [ ]:
import graphviz
from dataclasses import dataclass
from ast_parser import RegexParser, regex_parsen, Literal, Concatenation, Alternation, ast_to_dot, UnaryOp
from automaton import State, NFA, nfa_to_dot
from regex import regex_test, regex_find

## Zum Debuggen: Die Visualisierungsfunktion

Damit du siehst, was im Hintergrund passiert, nutzen wir diese Funktion. Sie zeichnet uns den AST (wie der Regex verstanden wurde) und den NFA (den daraus resultierenden Automaten).

In [ ]:
def regex_visualisieren(pattern, build_func=None, parse_func=regex_parsen):
    ast = parse_func(pattern)
    print(f"Muster: {pattern}")

    print("\n--- AST (Strukturbaum) ---")
    display(graphviz.Source(ast_to_dot(ast)))

    if build_func:
        nfa = build_func(ast, [0])
        print("\n--- NFA (Automat) ---")
        display(graphviz.Source(nfa_to_dot(nfa)))
    else:
        print("\n--- NFA noch nicht erstellt (keine build_func angegeben) ---")

## Schritt 1: Literale (Einzelzeichen)

Ein **Literal** ist das einfachste Element: ein einzelnes Zeichen, z.B. `'a'`.

Schau dir zuerst an, wie der Parser ein einzelnes Zeichen als AST darstellt:

In [ ]:
regex_visualisieren("a")

### Das Bauprinzip für Literale

Für ein Literal `a` erstellen wir zwei Zustände: einen Startzustand und einen akzeptierenden Endzustand. Wir verbinden sie mit einem Pfeil, der mit dem Zeichen `a` beschriftet ist.

In [ ]:
display(graphviz.Source(
    'digraph { rankdir=LR; node [shape=circle]; sStart [label="sStart", color=green]; sAccept [label="sAccept", shape="doublecircle" color=green]; sStart -> sAccept [label="Zeichen \'a\'", color=green]; }'))

Zunächst bereiten wir ein paar Funktionen vor, die wir später aufrufen werden:

- `neuer_zustand`: Wird mit dem `state_counter` aufgerufen, und erstellt einen neuen Zustand `sN`, wobei `N` der aktuelle Zählerwert ist.
- `nfa_bauen`: Erlaubt es uns einen Teil-NFA zusammen zu stellen. Die darin aufgerufene `knoten_transformieren`-Funktion werden wir Stück für Stück um die Implementierung erweitern.

In [ ]:
def neuer_zustand(counter):
    name = f"s{counter[0]}"
    counter[0] += 1
    return State(name)


def nfa_bauen(node, state_counter):
    if node is None:
        state = neuer_zustand(state_counter)
        return NFA(state, state)

    transformiert = knoten_transformieren(node, state_counter)

    if transformiert is None:
        raise TypeError(f"Diesen Knotentyp können wir noch nicht verarbeiten: {type(node)}")
    else:
        return transformiert

**Deine Aufgabe**: Implementiere den ersten Teil von `nfa_bauen`. Wir fangen nur mit `Literal` an.

In [ ]:
def literal_transformieren(node, state_counter=None):
    # TODO: Implementiere die Konstruktion für den Teil-NFA für ein Literal
    # 1. Neuen Startzustand "start" erstellen
    start = neuer_zustand(state_counter)
    # 2. Neuen Endzustand "accept" erstellen
    accept = neuer_zustand(state_counter)
    # 3. Übergang von start zu accept mit Zeichen `node.char` erstellen
    start.add_transition(node.char, accept)
    # 4. Teil-NFA zurückgeben
    return NFA(start, accept)


def knoten_transformieren(node, state_counter):
    if isinstance(node, Literal):
        return literal_transformieren(node, state_counter)
    return None


# Teste dein Ergebnis:
regex_visualisieren("a", build_func=nfa_bauen)

**Tipp:** Mit den Funktionen `regex_test` und `regex_find` kannst du deinen Automaten direkt an Beispielen ausprobieren:

In [ ]:
result = regex_test("a", "a", parse_func=regex_parsen, build_func=nfa_bauen)
print("Ganzer Text passt:", result)
regex_find("a", "blablubb", parse_func=regex_parsen, build_func=nfa_bauen)
print("Im Text enthalten:", result)

## Schritt 2: Konkatenation (Aneinanderreihen)

Wenn wir zwei Zeichen hintereinander haben (z.B. `ab`), nennt man das **Konkatenation**. Im AST siehst du dann einen `Concatenation`-Knoten, der zwei Unterbäume hat: Den Teilbaum für das Literal `a` (links) und den Teilbaum für das Literal `b` (rechts).

Schau dir den AST für `ab` an:

In [ ]:
regex_visualisieren("ab")

### Das Bauprinzip für Konkatenation

Um `AB` zu bauen, bauen wir erst den Automaten für `A` und den für `B`. Dann verbinden wir den Endzustand von `A` mit dem Startzustand von `B` über einen sogenannten **Epsilon-Übergang** (ein ε-Übergang, der kein Zeichen benötigt, wird in unserem Code durch `None` dargestellt).

In Grün sind die neu hinzugekommenen Zustände und Übergänge hervorgehoben:

In [ ]:
display(graphviz.Source(
    'digraph { rankdir=LR; node [shape=circle]; sStart [label="sStart", color=green]; sAccept [label="sAccept", color=green]; subgraph cluster_left { label="NFA links"; sL0 [label="Start links", color=blue]; sL1 [label="Ende links", shape=doublecircle]; sL0 -> sL1 [label="..."]; } subgraph cluster_right { label="NFA rechts"; sR0 [label="Start rechts", color=blue]; sR1 [label="Ende rechts", shape=doublecircle]; sR0 -> sR1 [label="..."]; } sStart -> sL0 [label="ε", color=green]; sL1 -> sR0 [label="ε (Epsilon)", color=green]; sR1 -> sAccept [label="ε", color=green]; }'))

**Deine Aufgabe**: Erweitere `nfa_bauen`, um `Concatenation` zu unterstützen.

In [ ]:
def concatenation_transformieren(node, state_counter):
    # TODO Implementiere die Konstruktion für den Teilbaum der Konkatenation:
    # 1. Start und Endzustand für den Teil-NFA definieren
    start = neuer_zustand(state_counter)
    accept = neuer_zustand(state_counter)
    # 2. Linken Teilbaum bauen (node.left)
    links = nfa_bauen(node.left, state_counter)
    # 3. Rechten Teilbaum bauen (node.right)
    rechts = nfa_bauen(node.right, state_counter)
    # 4. Den Endzustand von links (links.accept_state) mit dem Startzustand von rechts (rechts.start_state) verbinden
    links.accept_state.add_transition(None, rechts.start_state)
    # 5. Startzustand mit Startzustand von links verbinden
    start.add_transition(None, links.start_state)
    # 6. Endzustand von rechts mit Endzustand verbinden
    rechts.accept_state.add_transition(None, accept)
    # 7. Teil-NFA zurückgeben
    return NFA(start, accept)


def knoten_transformieren(node, state_counter):
    if isinstance(node, Literal):
        return literal_transformieren(node, state_counter)
    # TODO Konstruktion in unsere Transformation einhängen:
    elif isinstance(node, Concatenation):
        return concatenation_transformieren(node, state_counter)
    return None


# Teste dein Ergebnis:
regex_visualisieren("ab", build_func=nfa_bauen)

**Tipp**: Um den Automaten kleiner zu halten, kann man auch den Concatenation-Startzustand mit dem linken Startzustand, und den rechten Endzustand mit dem Concatenation-Endzustand zusammenlegen, und sich so den zusätzlichen `ε`-Übergang sparen.

## Schritt 3: Alternation (Oder / Auswahl)

Mit dem Zeichen `|` können wir sagen: "Entweder das oder das" (z.B. `a|b`). Im AST erscheint dies als `Alternation`-Knoten.

Schau dir den AST für `a|b` an:

In [ ]:
regex_visualisieren("a|b")

### Das Bauprinzip für Alternation

Für `A|B` erstellen wir einen neuen Startzustand, von dem aus wir per Epsilon-Übergang entweder zum Start von `A` oder zum Start von `B` springen können. Ebenso führen die Endzustände von `A` und `B` per Epsilon-Übergang zu einem gemeinsamen neuen Endzustand.

In [ ]:
display(graphviz.Source(
    'digraph { rankdir=LR; node [shape=circle]; sStart [label="sStart", color=green]; sAccept [label="sAccept", shape=doublecircle, color=green]; subgraph cluster_left { label="NFA links"; sL0 [label="Start links"]; sL1 [label="Ende links"]; sL0 -> sL1 [label="..."]; } subgraph cluster_right { label="NFA rechts"; sR0 [label="Start rechts"]; sR1 [label="Ende rechts"]; sR0 -> sR1 [label="..."]; } sStart -> sL0 [label="ε", color=green]; sStart -> sR0 [label="ε", color=green]; sL1 -> sAccept [label="ε", color=green]; sR1 -> sAccept [label="ε", color=green]; }'))

**Deine Aufgabe**: Erweitere `nfa_bauen` um `Alternation`.

In [ ]:
def alternation_transformieren(node, state_counter):
    # TODO Implementiere die Konstruktion für Alternation
    # 1. Start- und Endzustand für den Alternation-NFA definieren
    start = neuer_zustand(state_counter)
    accept = neuer_zustand(state_counter)
    # 2. Teil-NFAs für die beiden Pfade node.left und node.right bauen
    links = nfa_bauen(node.left, state_counter)
    rechts = nfa_bauen(node.right, state_counter)
    # 3. Startzustand mit beiden Teil-NFAs verbinden
    start.add_transition(None, links.start_state)
    start.add_transition(None, rechts.start_state)
    # 4. Teil-NFAs mit dem Endzustand verbinden
    links.accept_state.add_transition(None, accept)
    rechts.accept_state.add_transition(None, accept)
    # 5. Alternation-NFA zurückgeben
    return NFA(start, accept)


def knoten_transformieren(node, state_counter):
    if isinstance(node, Literal):
        return literal_transformieren(node, state_counter)
    elif isinstance(node, Concatenation):
        return concatenation_transformieren(node, state_counter)
    # TODO Transformation einhängen
    elif isinstance(node, Alternation):
        return alternation_transformieren(node, state_counter)
    return None


# Teste dein Ergebnis:
regex_visualisieren("a|b", build_func=nfa_bauen)

In [ ]:
regex_visualisieren("|a", build_func=nfa_bauen)

## Schritt 4: Der Kleene-Stern (Wiederholung)

Der Stern `*` bedeutet: "Nullmal oder beliebig oft". Im AST ist das ein `Star`-Knoten, der einen anderen Ausdruck umschließt.

Noch wird der Stern von unserem Parser aber als `Literal` interpretiert:

In [ ]:
regex_visualisieren("a*", build_func=nfa_bauen)

**Vorbereitung:** Erstmal passen wir also unseren Parser an.

Wir erstellen einen neuen unary-Operator `Star` und hängen ihn in den Parser ein:

In [ ]:
# Wir definieren den AST-Knoten für Star:
@dataclass
class Star(UnaryOp):
    pass


# Wir hängen den Stern als Postfix-Handler ein:
def regex_parsen(pattern):
    parser = RegexParser(pattern)
    parser.postfix_handlers['*'] = lambda node: Star(node)
    return parser.parse()

### Das Bauprinzip für den Stern

Für `A*` brauchen wir einen neuen Start- und Endzustand, um zwei Dinge zu ermöglichen:
1. **Wiederholung**: Der Endzustand von führt zurück zum Start.
2. **Überspringen**: Der Startzustand kann direkt zum Endzustand springen (wenn das Zeichen 0-mal vorkommt).

In [ ]:
display(graphviz.Source(
    'digraph { rankdir=LR; node [shape=circle]; sStart [label="sStart", color=green]; sAccept [label="sAccept", shape=doublecircle, color=green]; subgraph cluster_expr { label="NFA expression"; sL0 [label="Start expression", color=blue]; sL1 [label="Ende expression", shape=doublecircle]; sL0 -> sL1 [label="..."]; } sStart:e -> sL0:w [label="Eintritt", color=green]; sAccept:n -> sStart:n [label="Loop (Wiederholung)", color=lightgreen]; sL1 -> sAccept [label="Austritt", color=green]; sStart:s -> sAccept:s [label="Skip (0-mal)", color=lightgreen]; }'))

**Deine Aufgabe**: Vervollständige `nfa_bauen` für den `Star`-Knoten.

In [ ]:
def star_transformieren(node, state_counter):
    # TODO: Implementiere die Konstruktion für Star
    # Du bist auf dich allein gestellt! :)
    inner = nfa_bauen(node.expression, state_counter)
    start = neuer_zustand(state_counter)
    accept = neuer_zustand(state_counter)
    start.add_transition(None, inner.start_state)
    start.add_transition(None, accept)
    accept.add_transition(None, start)
    inner.accept_state.add_transition(None, accept)
    return NFA(start, accept)


def knoten_transformieren(node, state_counter):
    if isinstance(node, Literal):
        return literal_transformieren(node, state_counter)
    elif isinstance(node, Concatenation):
        return concatenation_transformieren(node, state_counter)
    elif isinstance(node, Alternation):
        return alternation_transformieren(node, state_counter)
    # Nicht vergessen, die Transformation einzuhängen!
    elif isinstance(node, Star):
        return star_transformieren(node, state_counter)
    return None


# Teste dein Ergebnis:
regex_visualisieren("a*", parse_func=regex_parsen, build_func=nfa_bauen)

## Finale Tests

Jetzt, wo du alle Grundbausteine hast, testen wir deinen Automaten mit verschiedenen Mustern und Texten.

In [ ]:
def regex_testen(pattern, text, expected):
    result = regex_test(pattern, text, parse_func=regex_parsen, build_func=nfa_bauen)
    print(f"Pattern: {pattern}, Text: {text}, Erwartet: {expected}, Ergebnis: {result}")
    assert result == expected


regex_testen("a", "a", True)
regex_testen("a", "b", False)
regex_testen("a|b", "a", True)
regex_testen("a|", "a", True)
regex_testen("a|b", "b", True)
regex_testen("a|b", "c", False)
regex_testen("a*", "", True)
regex_testen("a*", "aaaa", True)
regex_testen("a*", "b", False)
regex_testen("ab", "ab", True)
regex_testen("ab", "a", False)
regex_testen("(a|b)*c", "ababac", True)
regex_testen("(a|b)*c", "ababc", True)
regex_testen("(a|b)*c", "abab", False)
print("\nAlle Tests bestanden! Super gemacht!")

## Bonus: Den Parser erweitern

Unser Parser ist so gebaut, dass du ganz einfach neue Symbole hinzufügen kannst, ohne den Kerncode ändern zu müssen. Wir nutzen dafür die `postfix_handlers`.

### Aufgabe: Den `?` (Optional) Operator hinzufügen

Das Zeichen `?` bedeutet "null oder einmal".
1. Wir erstellen einen `Optional`-Knoten für unseren AST.
2. Wir sagen dem Parser, dass er das Zeichen `?` als `Optional`-Knoten behandeln soll.
3. Wir erweitern die NFA-Logik.

**Tipp 1**: Ein optionales `A?` ist fast wie `A*`, nur dass es keine Schleife zurück zum Start von `A` gibt.
**Tipp 2**: Ein optionales `A?` ist das selbe wie `(A|ε)`

In [ ]:
display(graphviz.Source(
    'digraph { rankdir=LR; node [shape=circle]; sStart [label="sStart", color=green]; sAccept [label="sAccept", shape=doublecircle, color=green]; subgraph cluster_A { label="NFA left"; sA0 [label="Start left"]; sA1 [label="Ende left"]; sA0 -> sA1 [label="..."]; } sStart:ne -> sA0:w [label="Eintritt", color=green]; sA1:e -> sAccept:nw [label="Austritt", color=green]; sStart:e -> sAccept:w [label="Skip (0-mal)", color=lightgreen]; }'))

Ohne Änderung an unserem Parser wird `?` einfach als Literal interpretiert:

In [ ]:
regex_visualisieren("a?b")

**Vorbereitung:** Auch hier erweitern wir also erstmal unseren Parser um das `?`-Zeichen als Postfix-Operator:

In [ ]:
def regex_parsen(pattern):
    parser = RegexParser(pattern)
    parser.postfix_handlers['*'] = lambda node: Star(node)
    # TODO: Parser für '?' erweitern:
    parser.postfix_handlers['?'] = lambda node: Alternation(node, None)
    return parser.parse()

Und sehen nun auch den neuen Knoten in unserem AST:

In [ ]:
regex_visualisieren("a?b(c|)", parse_func=regex_parsen)

## TODO: THIS EXAMPLE IS BROKEN (but not on the optional)

Fehlt nur noch die Implementierung in unserer Transformation:

In [ ]:
# Testen
regex_visualisieren("a?b", parse_func=regex_parsen, build_func=nfa_bauen)

In [ ]:
# TODO: Spielwiese

# Wenn du es bis hier geschafft hast: Wie wäre es damit, '+' zu implementieren?